# Social Media Content Planner Flow

In [1]:
import os
import yaml
import json
import requests
from pathlib import Path
from pydantic import BaseModel
from typing import Optional, List

from crewai import Agent, Task, Crew, LLM
# PERBAIKAN: Menghapus import 'router' yang ganda
from crewai.flow.flow import Flow, listen, start, router, or_
from crewai_tools import DirectoryReadTool, FileReadTool

from dotenv import load_dotenv
load_dotenv()

import nest_asyncio
nest_asyncio.apply()

In [2]:
llm = LLM(
    model="ollama/qwen3:0.6b-q4_K_M",
    base_url="http://localhost:11434"
)
blog_post_url = "https://huggingface.co/blog/llama4-release"

class Tweet(BaseModel):
    content: str
    is_hook: bool = False
    media_urls: Optional[List[str]] = []

class Thread(BaseModel):
    topic: str
    tweets: list[Tweet]

class LinkedInPost(BaseModel):
    content: str
    media_url: str

In [3]:
# Pastikan file config/planner_agents.yaml dan config/planner_tasks.yaml sudah ada
with open('config/planner_agents.yaml', 'r') as f:
    agents_config = yaml.safe_load(f)

with open('config/planner_tasks.yaml', 'r') as f:
    tasks_config = yaml.safe_load(f)

draft_analyzer = Agent(
    config=agents_config['draft_analyzer'], 
    tools=[DirectoryReadTool(), FileReadTool()], 
    llm=llm 
)

analyze_draft = Task(
    config=tasks_config['analyze_draft'],
    agent=draft_analyzer
)

# --- Twitter Crew ---
twitter_thread_planner = Agent(
    config=agents_config['twitter_thread_planner'], 
    tools=[DirectoryReadTool(), FileReadTool()], 
    llm=llm
)

create_twitter_thread_plan = Task(
    config=tasks_config['create_twitter_thread_plan'],
    agent=twitter_thread_planner,
    output_pydantic=Thread
)

twitter_planning_crew = Crew(
    agents=[draft_analyzer, twitter_thread_planner],
    tasks=[analyze_draft, create_twitter_thread_plan],
    verbose=True
)

# --- Linkedin Crew ---
linkedin_post_planner = Agent(
    config=agents_config['linkedin_post_planner'], 
    tools=[DirectoryReadTool(), FileReadTool()], 
    llm=llm
)

create_linkedin_post_plan = Task(
    config=tasks_config['create_linkedin_post_plan'],
    agent=linkedin_post_planner,
    output_pydantic=LinkedInPost
)

linkedin_planning_crew = Crew(
    agents=[draft_analyzer, linkedin_post_planner],
    tasks=[analyze_draft, create_linkedin_post_plan],
    verbose=True
)

### Defining State

In [ ]:
class ContentPlanningState(BaseModel):
    """
    State for the content planning flow
    """
    blog_post_url: str = blog_post_url
    draft_path: Path = Path("assets/")
    post_type: str = "twitter"
    path_to_example_threads: str = "assets/example_threads.txt"
    path_to_example_linkedin: str = "assets/example_linkedin.txt"

class ContentPlanningFlow(Flow[ContentPlanningState]):
    @start()
    def scrape_blog_post(self) -> ContentPlanningState:
        print(f"# fetching draft from: {self.state.blog_post_url}")
        
        jina_url = f"https://r.jina.ai/{self.state.blog_post_url}"
        
        try:
            # PERBAIKAN: Tambahkan timeout 30 detik untuk menghindari hang/httpcore error
            response = requests.get(jina_url, timeout=30)
            response.raise_for_status() # Cek jika ada error HTTP (404/500)
            markdown_content = response.text
        except requests.exceptions.RequestException as e:
            print(f"Error fetching URL: {e}")
            markdown_content = "Error retrieving content."

        # Pembersihan nama file
        clean_url = self.state.blog_post_url.rstrip('/')
        title = clean_url.split('/')[-1]
        if not title:
            title = "downloaded_draft"
        title = title.split('?')[0].split('#')[0]
        
        os.makedirs("assets", exist_ok=True)
        
        # Simpan path sebagai string atau Path object sesuai kebutuhan tools
        self.state.draft_path = Path(f'assets/{title}.md')
        
        with open(self.state.draft_path, 'w', encoding='utf-8') as f:
            f.write(markdown_content)
        
        print(f"File saved to: {self.state.draft_path}")
        return self.state

    @router(scrape_blog_post)
    def select_platform(self) -> str:
        if self.state.post_type == "twitter":
            return "twitter"
        elif self.state.post_type == "linkedin":
            return "linkedin"
        else:
            print(f"Unknown platform: {self.state.post_type}, defaulting to twitter")
            return "twitter"

    @listen("twitter")
    def twitter_draft(self):
        print(f"# Planning content for: {self.state.draft_path}")
        
        # Pastikan path dikonversi ke string jika tools membutuhkannya
        result = twitter_planning_crew.kickoff(
            inputs={
                'draft_path': str(self.state.draft_path), 
                'path_to_example_threads': self.state.path_to_example_threads
            }
        )
        
        print(f"# Planned content for {self.state.draft_path}:")
        
        # Handling output jika berupa object Pydantic atau dict
        tweets = result.pydantic.tweets if hasattr(result, 'pydantic') and result.pydantic else result.raw
        
        # Jika hasil sukses diparsing ke Pydantic
        if hasattr(result, 'pydantic') and result.pydantic:
            for i, tweet in enumerate(tweets):
                print(f"Tweet {i+1}:")
                print(f"{tweet.content}")
                print(f"Media URLs: {tweet.media_urls}")
                print("-"*100)
        else:
            print(result.raw)
            
        return result

    @listen("linkedin")
    def linkedin_draft(self):
        print(f"# Planning content for: {self.state.draft_path}")
        result = linkedin_planning_crew.kickoff(
            inputs={
                'draft_path': str(self.state.draft_path),  # PERBAIKAN: Konversi Path ke string
                'path_to_example_linkedin': self.state.path_to_example_linkedin
            }
        )
        print(f"# Planned content for {self.state.draft_path}:")
        
        content = result.pydantic.content if hasattr(result, 'pydantic') and result.pydantic else result.raw
        print(content)
        return result

    @listen(or_(twitter_draft, linkedin_draft))
    def save_plan(self, plan):
        # Pastikan direktori output ada
        os.makedirs("output", exist_ok=True)
        
        output_filename = f'output/{str(self.state.draft_path).split("/")[-1]}_{self.state.post_type}.json'
        
        data_to_save = {}
        if hasattr(plan, 'pydantic') and plan.pydantic:
            data_to_save = plan.pydantic.model_dump()
        else:
            data_to_save = {"raw_output": plan.raw}

        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(data_to_save, f, indent=2)
        
        print(f"Plan saved to {output_filename}")

### Run Flow

In [5]:
flow = ContentPlanningFlow()

print(">>> Running for LinkedIn...")
flow.state.post_type = "linkedin"

>>> Running for LinkedIn...


In [6]:
flow.plot()

'/tmp/crewai_flow_x63186ua/crewai_flow.html'

Opening in existing browser session.


In [ ]:
flow.kickoff()